In [6]:
import pandas as pd
orders = pd.DataFrame({
    "order_id": [101, 102, 103],
    "customer_id": ["C01", "C02", "C01"],
    "amount": [100, 200, 150],
})

customer_regions = pd.DataFrame({
    "customer_id": ["C01", "C01", "C02"],
    "region": ["North", "Central", "South"],
})

In [7]:
print("Is customer IDs in customer region unique: ",customer_regions["customer_id"].is_unique)

Is customer IDs in customer region unique:  False


### Attempting a Many-to-One Merge

```python
customer_orders = orders.merge(
    customer_regions,
    on="customer_id",
    how="left",
    validate="many_to_one"
)
```

### What Does `validate="many_to_one"` Mean?

It tells pandas to verify that:

- The **left DataFrame** (`orders`) may contain duplicate `customer_id` values.
- The **right DataFrame** (`customer_regions`) must contain unique `customer_id` values.

```md
Many Orders  →  One Customer Record
```

### Why Does This Merge Fail?

The `customer_regions` table contains duplicate customer IDs:

| customer_id | region |
|------------|---------|
| C01 | North |
| C01 | Central |
| C02 | South |

Notice that **C01 appears twice**.

Therefore:

```python
customer_regions["customer_id"].is_unique
```

returns:

```python
False
```

Because the right table does not have unique `customer_id` values, the relationship is **not many-to-one**.

### Error Produced

Pandas raises a `MergeError` similar to:

```text
MergeError:
Merge keys are not unique in right dataset;
not a many-to-one merge
```

### Why Is This Validation Useful?

Without validation, pandas would perform the merge and duplicate rows:

| order_id | customer_id | amount | region |
|----------|-------------|---------|---------|
| 101 | C01 | 100 | North |
| 101 | C01 | 100 | Central |
| 102 | C02 | 200 | South |
| 103 | C01 | 150 | North |
| 103 | C01 | 150 | Central |

This may inflate totals and produce incorrect business results.

Using `validate="many_to_one"` helps detect data quality issues before analysis.

### Conclusion

- The merge **fails** because `customer_regions` contains duplicate `customer_id` values.
- A many-to-one merge requires the right-hand table to have unique keys.
- The validation check correctly prevents an invalid merge and alerts you to the problem.